# Model Training (Independent)


This notebook is an independent version of the final training pipeline.

It does four things:

1. builds a spatially grouped evaluation pipeline (KMeans `spatial_group` + pseudo-holdout buffer logic),
2. runs scout and full-stage target-specific model sweeps,
3. freezes safety/challenger manifests from grouped CV results,
4. writes four submission files:
   - **A** = safe anchor,
   - **B** = modest challenger blend,
   - **C** = hedge blend,
   - **D** = DRP soft-hedge variant.

Notes:
- Environment/config logic is inlined in this notebook (`ENV`, `MLFLOW_URI`, `load_data`, `save_artifacts`).
- Sentinel train/test parquet paths are loaded directly (no aligned fallback path logic).


In [ ]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from IPython.display import display
from tqdm.auto import tqdm


## Environment and MLflow

In [ ]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    MLFLOW_URI = 'sqlite:///../mlflow.db'

    def load_data():
        # Reads from local parquet file.
        return pd.read_parquet('../data/interim/master_train.parquet')

    def save_artifacts(preprocessor, model, run_name):
        os.makedirs('../models', exist_ok=True)
        joblib.dump(preprocessor, f'../models/preprocessor_{run_name}.joblib')
        joblib.dump(model, f'../models/model_{run_name}.joblib')
        print('Artifacts saved locally.')
else:
    MLFLOW_URI = 'sqlite:////tmp/mlflow.db'

    def load_data():
        from snowflake.snowpark.context import get_active_session

        session = get_active_session()
        # Reads directly from the Snowflake secure stage.
        return session.read.parquet('@ML_DATA_STAGE/integrated_features.parquet').to_pandas()

    def save_artifacts(preprocessor, model, run_name):
        from snowflake.snowpark.context import get_active_session

        session = get_active_session()
        os.makedirs('/tmp/models', exist_ok=True)
        joblib.dump(preprocessor, f'/tmp/models/preprocessor_{run_name}.joblib')
        joblib.dump(model, f'/tmp/models/model_{run_name}.joblib')

        # Push local artifacts to Snowflake stage.
        session.sql('CREATE STAGE IF NOT EXISTS @ML_ARTIFACTS_STAGE').collect()
        session.file.put('file:///tmp/mlflow.db', '@ML_ARTIFACTS_STAGE/mlflow/', auto_compress=False, overwrite=True)
        session.file.put('file:///tmp/models/*.joblib', '@ML_ARTIFACTS_STAGE/models/', auto_compress=False, overwrite=True)
        print('Artifacts secured in Snowflake Stage.')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', MLFLOW_URI)


## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [ ]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'SpatialV2_GroupKFold+ContiguousHoldout+Buffer'
GROUP_DEFINITION_VERSION = 'kmeans_latlon_v3_contiguous_holdout_buffer'
PIPELINE_VERSION = 'deadline_v3_spatial_v2_holdout_buffer'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline_mvp4'

SPATIAL_N_CLUSTERS = 16
CV_N_SPLITS = 5
HOLDOUT_MARGIN_DEG = 0.25
HOLDOUT_BUFFER_DEG = 0.18
HOLDOUT_MIN_GROUPS = 3
HOLDOUT_MIN_FRAC = 0.08
HOLDOUT_MAX_FRAC = 0.15

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

## Data loading

Load sentinel train/validation parquet files directly:
- `../data/interim/master_train_sentinel.parquet`
- `../data/interim/master_test_sentinel.parquet`

This stage also:
- validates required geo/time/target columns,
- applies optional contract checks,
- creates KMeans `spatial_group`,
- builds pseudo-holdout and buffer-exclusion masks for spatial v2 evaluation.


In [ ]:
TRAIN_PATH = '../data/interim/master_train_sentinel.parquet'
VALID_PATH = '../data/interim/master_test_sentinel.parquet'
CONTRACT_TXT_PATH = '../data/interim/feature_contract_master_sentinel.txt'

if not os.path.exists(TRAIN_PATH):
    raise RuntimeError(f'TRAIN_PATH not found: {TRAIN_PATH}')
if not os.path.exists(VALID_PATH):
    raise RuntimeError(f'VALID_PATH not found: {VALID_PATH}')

# Train and validation data are prepared and can be loaded directly.
df = pd.read_parquet(TRAIN_PATH).copy()
df_val_all = pd.read_parquet(VALID_PATH).copy()

required_cols = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
missing_cols_train = [c for c in required_cols if c not in df.columns]
if missing_cols_train:
    raise RuntimeError(f'Missing required training columns: {missing_cols_train}')

missing_cols_valid_geo = [c for c in ['Latitude', 'Longitude', 'Sample Date'] if c not in df_val_all.columns]
if missing_cols_valid_geo:
    raise RuntimeError(f'Missing required validation geo/time columns: {missing_cols_valid_geo}')

# Optional schema-contract guard (produced by 01_eda_and_discovery / 02_preprocessing)
if os.path.exists(CONTRACT_TXT_PATH):
    with open(CONTRACT_TXT_PATH, 'r', encoding='utf-8') as f:
        contract_cols = [line.strip() for line in f.readlines() if line.strip()]

    missing_contract_train = [c for c in contract_cols if c not in df.columns]
    missing_contract_valid = [c for c in contract_cols if c not in df_val_all.columns]

    if missing_contract_train:
        raise RuntimeError(f'Contract columns missing in train ({len(missing_contract_train)}): {missing_contract_train[:20]}')
    if missing_contract_valid:
        raise RuntimeError(f'Contract columns missing in validation ({len(missing_contract_valid)}): {missing_contract_valid[:20]}')

    print(f'Contract check passed: {len(contract_cols)} feature columns present in train/validation.')
else:
    print('Contract TXT not found; continuing without contract schema guard.')


df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce')
df_val_geo = df_val_all[['Latitude', 'Longitude', 'Sample Date']].copy()
df_val_geo['Sample Date'] = pd.to_datetime(df_val_geo['Sample Date'], errors='coerce')


def add_spatial_groups(data: pd.DataFrame, n_clusters: int = SPATIAL_N_CLUSTERS) -> pd.DataFrame:
    '''
    Create stable spatial groups from latitude/longitude using KMeans.
    '''
    out = data.copy()
    n_clusters = min(max(4, int(n_clusters)), len(out))

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    out['spatial_group'] = km.fit_predict(out[['Latitude', 'Longitude']].astype(float)).astype(str)
    return out


def select_pseudo_holdout_groups(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    min_groups: int = HOLDOUT_MIN_GROUPS,
    min_frac: float = HOLDOUT_MIN_FRAC,
    max_frac: float = HOLDOUT_MAX_FRAC,
    margin_deg: float = HOLDOUT_MARGIN_DEG,
):
    '''
    Spatial v2 holdout:
    1) Start from groups whose centroids fall inside validation bbox (+margin),
    2) Expand contiguously by nearest-centroid growth,
    3) Respect target holdout fraction bounds.
    '''
    gdf = train_df.groupby('spatial_group', as_index=False).agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        n=('spatial_group', 'size')
    )

    lat_min = float(valid_df['Latitude'].min()) - margin_deg
    lat_max = float(valid_df['Latitude'].max()) + margin_deg
    lon_min = float(valid_df['Longitude'].min()) - margin_deg
    lon_max = float(valid_df['Longitude'].max()) + margin_deg

    valid_center_lat = float(valid_df['Latitude'].mean())
    valid_center_lon = float(valid_df['Longitude'].mean())

    gdf['Latitude'] = gdf['Latitude'].astype(float)
    gdf['Longitude'] = gdf['Longitude'].astype(float)
    gdf['center_dist'] = np.sqrt(
        (gdf['Latitude'] - valid_center_lat) ** 2 +
        (gdf['Longitude'] - valid_center_lon) ** 2
    )

    in_bbox = (
        gdf['Latitude'].between(lat_min, lat_max) &
        gdf['Longitude'].between(lon_min, lon_max)
    )

    seed = gdf.loc[in_bbox].sort_values(['center_dist', 'n'], ascending=[True, False]).copy()
    if seed.empty:
        seed = gdf.sort_values(['center_dist', 'n'], ascending=[True, False]).head(1).copy()

    total_rows = int(len(train_df))
    selected = []
    selected_rows = 0

    for _, row in seed.iterrows():
        g = str(row['spatial_group'])
        if g not in selected:
            selected.append(g)
            selected_rows += int(row['n'])

    def holdout_frac(rows_count):
        return rows_count / max(1, total_rows)

    while len(selected) < int(min_groups) or holdout_frac(selected_rows) < float(min_frac):
        selected_df = gdf[gdf['spatial_group'].astype(str).isin(selected)].copy()
        remaining_df = gdf[~gdf['spatial_group'].astype(str).isin(selected)].copy()
        if remaining_df.empty:
            break

        # Contiguous growth: add the group closest to current selected centroid cloud.
        s_lat = selected_df['Latitude'].to_numpy(dtype=float)
        s_lon = selected_df['Longitude'].to_numpy(dtype=float)
        r_lat = remaining_df['Latitude'].to_numpy(dtype=float)
        r_lon = remaining_df['Longitude'].to_numpy(dtype=float)

        dists = []
        for i in range(len(remaining_df)):
            dd = np.sqrt((s_lat - r_lat[i]) ** 2 + (s_lon - r_lon[i]) ** 2)
            dists.append(float(np.min(dd)))

        remaining_df = remaining_df.assign(adj_dist=np.asarray(dists, dtype=float))
        remaining_df = remaining_df.sort_values(['adj_dist', 'center_dist', 'n'], ascending=[True, True, False])

        picked = None
        for _, cand in remaining_df.iterrows():
            next_rows = selected_rows + int(cand['n'])
            must_add = len(selected) < int(min_groups) or holdout_frac(selected_rows) < float(min_frac)
            if must_add or holdout_frac(next_rows) <= float(max_frac):
                picked = cand
                break

        if picked is None:
            break

        g = str(picked['spatial_group'])
        selected.append(g)
        selected_rows += int(picked['n'])

    return selected


def assign_spatial_v2_masks(
    train_df: pd.DataFrame,
    holdout_groups: list,
    buffer_deg: float = HOLDOUT_BUFFER_DEG,
) -> pd.DataFrame:
    '''
    Build split masks for spatial v2.
    - is_pseudo_valid: holdout groups
    - is_buffer_excluded: non-holdout rows too close to holdout bbox
    - is_spatial_v2_active: rows used by CV/selection
    '''
    out = train_df.copy()
    holdout_set = set(pd.Series(holdout_groups).astype(str).tolist())
    out['is_pseudo_valid'] = out['spatial_group'].astype(str).isin(holdout_set)

    out['is_buffer_excluded'] = False
    if float(buffer_deg) > 0 and out['is_pseudo_valid'].any():
        h = out.loc[out['is_pseudo_valid'], ['Latitude', 'Longitude']].copy()
        h_lat_min = float(h['Latitude'].min())
        h_lat_max = float(h['Latitude'].max())
        h_lon_min = float(h['Longitude'].min())
        h_lon_max = float(h['Longitude'].max())

        in_buffer_box = (
            out['Latitude'].astype(float).between(h_lat_min - buffer_deg, h_lat_max + buffer_deg) &
            out['Longitude'].astype(float).between(h_lon_min - buffer_deg, h_lon_max + buffer_deg)
        )
        out['is_buffer_excluded'] = (~out['is_pseudo_valid']) & in_buffer_box

    out['is_spatial_v2_active'] = ~out['is_buffer_excluded']

    if (out['is_pseudo_valid'] & out['is_buffer_excluded']).any():
        raise RuntimeError('Invalid split masks: holdout rows were buffer-excluded.')

    return out


# Build spatial groups and spatial-v2 holdout masks
# This must run before grouped_oof_eval (which expects `spatial_group`).
df = add_spatial_groups(df, n_clusters=SPATIAL_N_CLUSTERS)

holdout_groups = select_pseudo_holdout_groups(
    train_df=df,
    valid_df=df_val_geo,
    min_groups=HOLDOUT_MIN_GROUPS,
    min_frac=HOLDOUT_MIN_FRAC,
    max_frac=HOLDOUT_MAX_FRAC,
    margin_deg=HOLDOUT_MARGIN_DEG,
)

if not holdout_groups:
    raise RuntimeError('Pseudo-holdout selection returned no groups.')

df = assign_spatial_v2_masks(
    train_df=df,
    holdout_groups=holdout_groups,
    buffer_deg=HOLDOUT_BUFFER_DEG,
)

holdout_group_set = set(pd.Series(holdout_groups).astype(str).tolist())

print('Spatial v2 grouping ready.')
print('n_rows_total:', len(df))
print('n_rows_active:', int(df['is_spatial_v2_active'].sum()))
print('n_buffer_excluded:', int(df['is_buffer_excluded'].sum()))
print('n_spatial_groups:', int(df['spatial_group'].nunique()))
print('holdout_groups:', sorted(list(holdout_group_set)))
print('holdout_rows:', int(df['is_pseudo_valid'].sum()), '| holdout_frac:', round(float(df['is_pseudo_valid'].mean()), 4))

# Guard against group leakage between pseudo-holdout and train subsets (active rows only)
train_groups = set(df.loc[df['is_spatial_v2_active'] & (~df['is_pseudo_valid']), 'spatial_group'].astype(str).tolist())
test_groups = set(df.loc[df['is_spatial_v2_active'] & (df['is_pseudo_valid']), 'spatial_group'].astype(str).tolist())
if train_groups.intersection(test_groups):
    raise RuntimeError('Pseudo-holdout leakage detected after split setup.')
if len(train_groups) < 2 or len(test_groups) < 1:
    raise RuntimeError('Spatial v2 split invalid: insufficient active train/holdout groups.')


In [ ]:
# OPTIONAL DIAGNOSTIC (disabled for faster runs)
# print(df.info())
# Uncomment above if you need full schema summary before training.


## Feature engineering

This notebook applies high-ROI engineered features from the current schema, including:

- temporal cyclic features,
- hydroclimate interactions,
- soil/hydrology/population interactions,
- OSM proximity/pressure transforms,
- SANLC aggregate shares and deltas.

Per-class SANLC deltas are available behind a toggle for ablation runs.


In [ ]:
# Feature-engineering ablation toggles
FE_INCLUDE_CLASS_DELTAS = False  # quick ablation: disable noisy per-class SANLC deltas


def _safe_div(a: pd.Series, b: pd.Series, eps: float = 1e-6) -> pd.Series:
    return a.astype(float) / (b.astype(float) + eps)


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    High-ROI engineered features from current schema:
    - temporal cyclics (if date/month exists)
    - hydroclimate interactions (Terra + weather)
    - soil/hydro/population interactions
    - OSM pressure/proximity transforms + interactions
    - SANLC thematic aggregate shares + deltas
    Optional:
    - per-class SANLC deltas (controlled by FE_INCLUDE_CLASS_DELTAS)
    """
    out = data.copy()

    # Clean previously engineered columns if function is re-run in same kernel state.
    exact_drop = {
        'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'is_wet_season',
        'aridity_idx', 'water_balance', 'evap_eff', 'runoff_ratio', 'dry_heat', 'temp_range',
        'rain_wind_event', 'soil_texture_balance', 'soil_fines', 'drainage_proxy',
        'upstream_human_pressure', 'stream_power_proxy', 'river_discharge_per_width',
        'hydro_pressure_index', 'pop_compaction_1km', 'pop_gradient', 'pop_sum_ratio_1_to_5km',
        'human_land_share_2020', 'human_land_share_2022', 'human_land_share_delta',
        'osm_total_pressure', 'farm_rain', 'farm_runoff', 'mine_runoff', 'ww_dry',
        'ww_urban', 'farm_crop', 'mine_land',
    }
    drop_cols = [
        c for c in out.columns
        if c in exact_drop
        or c.startswith('sanlc_delta_')
        or c.startswith('sanlc_abs_delta_')
        or c.startswith('osm_log_count_')
        or c.startswith('osm_log_dist_')
        or c.startswith('osm_inv_dist_')
        or c.startswith('osm_local_ratio_')
        or c.startswith('osm_ring_count_')
        or c.startswith('osm_pressure_')
        or c.startswith('osm_near_')
        or (c.startswith('sanlc_') and ('_share_2020' in c or '_share_2022' in c or c.endswith('_delta') or '_ratio_' in c))
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols, errors='ignore')

    new_cols = {}

    # -------------------------
    # 1) Temporal cyclic features
    # -------------------------
    month = None
    doy = None

    date_col = None
    for cand in ['Sample_Date', 'Sample Date', 'Date', 'date']:
        if cand in out.columns:
            date_col = cand
            break

    if date_col is not None:
        dt = pd.to_datetime(out[date_col], errors='coerce')
        month = dt.dt.month.astype(float)
        doy = dt.dt.dayofyear.astype(float)
    elif 'Month' in out.columns:
        month = pd.to_numeric(out['Month'], errors='coerce').astype(float)
    elif 'month' in out.columns:
        month = pd.to_numeric(out['month'], errors='coerce').astype(float)

    if month is not None:
        new_cols['month_sin'] = np.sin(2.0 * np.pi * (month / 12.0))
        new_cols['month_cos'] = np.cos(2.0 * np.pi * (month / 12.0))
        wet_months = {11, 12, 1, 2, 3}
        new_cols['is_wet_season'] = np.where(month.isna(), np.nan, month.isin(wet_months).astype(float))

    if doy is not None:
        new_cols['doy_sin'] = np.sin(2.0 * np.pi * (doy / 366.0))
        new_cols['doy_cos'] = np.cos(2.0 * np.pi * (doy / 366.0))

    # -------------------------
    # 2) Hydroclimate interactions (Terra + weather)
    # -------------------------
    if {'terra_pet', 'terra_ppt'}.issubset(out.columns):
        pet = out['terra_pet'].fillna(0.0)
        ppt = out['terra_ppt'].fillna(0.0)
        new_cols['aridity_idx'] = _safe_div(pet, ppt)
        new_cols['water_balance'] = ppt - pet

    if {'terra_aet', 'terra_pet'}.issubset(out.columns):
        new_cols['evap_eff'] = _safe_div(
            out['terra_aet'].fillna(0.0),
            out['terra_pet'].fillna(0.0),
        )

    if {'terra_q', 'terra_ppt'}.issubset(out.columns):
        new_cols['runoff_ratio'] = _safe_div(
            out['terra_q'].fillna(0.0),
            out['terra_ppt'].fillna(0.0),
        )

    if {'terra_vpd', 'terra_tmax'}.issubset(out.columns):
        new_cols['dry_heat'] = (
            out['terra_vpd'].fillna(0.0).astype(float)
            * out['terra_tmax'].fillna(0.0).astype(float)
        )

    if {'terra_tmax', 'terra_tmin'}.issubset(out.columns):
        new_cols['temp_range'] = (
            out['terra_tmax'].fillna(0.0).astype(float)
            - out['terra_tmin'].fillna(0.0).astype(float)
        )

    if {'weather_precip_7d_sum', 'weather_wind_7d_mean'}.issubset(out.columns):
        new_cols['rain_wind_event'] = (
            out['weather_precip_7d_sum'].fillna(0.0).astype(float)
            * out['weather_wind_7d_mean'].fillna(0.0).astype(float)
        )

    # -------------------------
    # 3) Soil / hydro / population interactions
    # -------------------------
    if {'soil_sand_mean_0_5cm', 'soil_clay_mean_0_5cm'}.issubset(out.columns):
        new_cols['soil_texture_balance'] = (
            out['soil_sand_mean_0_5cm'].fillna(0.0).astype(float)
            - out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        )

    if {'soil_clay_mean_0_5cm', 'soil_silt_mean_0_5cm'}.issubset(out.columns):
        clay = out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        silt = out['soil_silt_mean_0_5cm'].fillna(0.0).astype(float)
        new_cols['soil_fines'] = clay + silt

    if {'soil_sand_mean_0_5cm', 'soil_clay_mean_0_5cm', 'soil_silt_mean_0_5cm'}.issubset(out.columns):
        sand = out['soil_sand_mean_0_5cm'].fillna(0.0).astype(float)
        clay = out['soil_clay_mean_0_5cm'].fillna(0.0).astype(float)
        silt = out['soil_silt_mean_0_5cm'].fillna(0.0).astype(float)
        new_cols['drainage_proxy'] = _safe_div(sand, clay + silt)

    if {'basin_population', 'basin_upstream_area_km2'}.issubset(out.columns):
        pop = out['basin_population'].fillna(0.0).astype(float).clip(lower=0.0)
        area = out['basin_upstream_area_km2'].fillna(0.0).astype(float).clip(lower=0.0)
        new_cols['upstream_human_pressure'] = np.log1p(pop) / (np.log1p(area) + 1e-6)

    if {'dem_slope_1km', 'river_avg_discharge_cms'}.issubset(out.columns):
        slope = out['dem_slope_1km'].fillna(0.0).astype(float)
        q = out['river_avg_discharge_cms'].fillna(0.0).astype(float)
        new_cols['stream_power_proxy'] = slope * q

    if {'river_avg_discharge_cms', 'river_width_m'}.issubset(out.columns):
        new_cols['river_discharge_per_width'] = _safe_div(
            out['river_avg_discharge_cms'].fillna(0.0).astype(float),
            out['river_width_m'].fillna(0.0).astype(float),
        )

    stream_power_proxy = new_cols.get('stream_power_proxy')
    upstream_human_pressure = new_cols.get('upstream_human_pressure')
    if stream_power_proxy is not None and upstream_human_pressure is not None:
        new_cols['hydro_pressure_index'] = (
            stream_power_proxy.fillna(0.0).astype(float)
            * upstream_human_pressure.fillna(0.0).astype(float)
        )

    if {'worldpop_max_1km', 'worldpop_mean_1km'}.issubset(out.columns):
        new_cols['pop_compaction_1km'] = _safe_div(
            out['worldpop_max_1km'].fillna(0.0).astype(float),
            out['worldpop_mean_1km'].fillna(0.0).astype(float),
        )

    if {'worldpop_mean_1km', 'worldpop_mean_5km'}.issubset(out.columns):
        new_cols['pop_gradient'] = (
            out['worldpop_mean_1km'].fillna(0.0).astype(float)
            - out['worldpop_mean_5km'].fillna(0.0).astype(float)
        )

    if {'worldpop_sum_1km', 'worldpop_sum_5km'}.issubset(out.columns):
        new_cols['pop_sum_ratio_1_to_5km'] = _safe_div(
            out['worldpop_sum_1km'].fillna(0.0).astype(float),
            out['worldpop_sum_5km'].fillna(0.0).astype(float),
        )

    # -------------------------
    # 4) OSM transforms and source pressure
    # -------------------------
    osm_map = {
        'mine': ('osm_dist_nearest_mine', 'osm_mine_count_1km', 'osm_mine_count_5km'),
        'farm': ('osm_dist_nearest_farm', 'osm_farm_count_1km', 'osm_farm_count_5km'),
        'wastewater': ('osm_dist_nearest_wastewater', 'osm_wastewater_count_1km', 'osm_wastewater_count_5km'),
    }

    for src, (dist_col, c1_col, c5_col) in osm_map.items():
        has_dist = dist_col in out.columns
        has_c1 = c1_col in out.columns
        has_c5 = c5_col in out.columns

        if has_dist:
            dist = out[dist_col].fillna(0.0).astype(float).clip(lower=0.0)
            new_cols[f'osm_log_dist_{src}'] = np.log1p(dist)
            new_cols[f'osm_inv_dist_{src}'] = 1.0 / (1.0 + dist)
            new_cols[f'osm_near_{src}_lt1km'] = (dist <= 1000.0).astype(float)
            new_cols[f'osm_near_{src}_lt5km'] = (dist <= 5000.0).astype(float)

        if has_c1:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            new_cols[f'osm_log_count_{src}_1km'] = np.log1p(c1)

        if has_c5:
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            new_cols[f'osm_log_count_{src}_5km'] = np.log1p(c5)

        if has_c1 and has_c5:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            new_cols[f'osm_local_ratio_{src}'] = _safe_div(c1, c5 + 1.0)
            new_cols[f'osm_ring_count_{src}'] = (c5 - c1).clip(lower=0.0)

        if has_dist and has_c1 and has_c5:
            c1 = out[c1_col].fillna(0.0).astype(float).clip(lower=0.0)
            c5 = out[c5_col].fillna(0.0).astype(float).clip(lower=0.0)
            inv_d = new_cols[f'osm_inv_dist_{src}']
            new_cols[f'osm_pressure_{src}'] = np.log1p(c1) + 0.5 * np.log1p(c5) + inv_d

    pressure_cols = [f'osm_pressure_{s}' for s in ['mine', 'farm', 'wastewater'] if f'osm_pressure_{s}' in new_cols]
    if pressure_cols:
        new_cols['osm_total_pressure'] = pd.concat(
            [new_cols[c] for c in pressure_cols], axis=1
        ).sum(axis=1)

    if 'osm_pressure_farm' in new_cols and 'weather_precip_7d_sum' in out.columns:
        new_cols['farm_rain'] = new_cols['osm_pressure_farm'].fillna(0.0) * out['weather_precip_7d_sum'].fillna(0.0)

    if 'osm_pressure_farm' in new_cols and 'terra_q' in out.columns:
        new_cols['farm_runoff'] = new_cols['osm_pressure_farm'].fillna(0.0) * out['terra_q'].fillna(0.0)

    if 'osm_pressure_mine' in new_cols and 'terra_q' in out.columns:
        new_cols['mine_runoff'] = new_cols['osm_pressure_mine'].fillna(0.0) * out['terra_q'].fillna(0.0)

    if 'osm_pressure_wastewater' in new_cols:
        if 'aridity_idx' in new_cols:
            dry_ref = new_cols['aridity_idx'].fillna(0.0)
        elif 'terra_vpd' in out.columns:
            dry_ref = out['terra_vpd'].fillna(0.0)
        else:
            dry_ref = None
        if dry_ref is not None:
            new_cols['ww_dry'] = new_cols['osm_pressure_wastewater'].fillna(0.0) * dry_ref

    # -------------------------
    # 5) SANLC aggregate themes + deltas
    # -------------------------
    p20 = 'sanlc2020_pct_'
    p22 = 'sanlc2022_pct_'

    cols20 = [c for c in out.columns if c.startswith(p20)]
    cols22 = [c for c in out.columns if c.startswith(p22)]

    suffix20 = {c[len(p20):] for c in cols20}
    suffix22 = {c[len(p22):] for c in cols22}
    common_suffixes = sorted(suffix20.intersection(suffix22))

    if FE_INCLUDE_CLASS_DELTAS:
        for suf in common_suffixes:
            c20 = p20 + suf
            c22 = p22 + suf
            v20 = out[c20].fillna(0.0).astype(float)
            v22 = out[c22].fillna(0.0).astype(float)
            delta = v22 - v20
            new_cols[f'sanlc_delta_{suf}'] = delta
            new_cols[f'sanlc_abs_delta_{suf}'] = delta.abs()

    theme_keywords = {
        'urban': ['urban', 'residential', 'village', 'settlement', 'roads', 'rails', 'industrial', 'built'],
        'mining': ['mine', 'mines', 'tailings', 'resource_dumps', 'quarr', 'extraction_pits'],
        'cropland': ['crop', 'crops', 'cultivated', 'orchard', 'vineyard', 'fallow', 'smallholding', 'sugarcane'],
        'wetland': ['wetland', 'wetlands', 'marsh', 'peat'],
        'water': ['river', 'rivers', 'dam', 'dams', 'canal', 'water', 'estuar', 'lagoon', 'pans'],
        'bare': ['bare', 'rock', 'riverbed', 'sand'],
        'forest': ['forest', 'woodland', 'thicket'],
        'grass': ['grassland', 'grass', 'herbaceous', 'shrubland', 'bush'],
    }

    def cols_for_theme(year_cols, keywords):
        selected = []
        for c in year_cols:
            cl = c.lower()
            if any(k in cl for k in keywords):
                selected.append(c)
        return selected

    for year, prefix in [('2020', p20), ('2022', p22)]:
        ycols = [c for c in out.columns if c.startswith(prefix)]
        if ycols:
            total_share = out[ycols].fillna(0.0).sum(axis=1)
            new_cols[f'sanlc_total_share_{year}'] = total_share

        for theme, keywords in theme_keywords.items():
            tcols = cols_for_theme(ycols, keywords)
            if tcols:
                raw_share = out[tcols].fillna(0.0).sum(axis=1)
                new_cols[f'sanlc_{theme}_share_{year}'] = raw_share
                if ycols:
                    new_cols[f'sanlc_{theme}_ratio_{year}'] = _safe_div(
                        raw_share,
                        new_cols[f'sanlc_total_share_{year}'].fillna(0.0),
                    )

    for theme in list(theme_keywords.keys()) + ['total']:
        c20 = f'sanlc_{theme}_share_2020'
        c22 = f'sanlc_{theme}_share_2022'
        if c20 in new_cols and c22 in new_cols:
            new_cols[f'sanlc_{theme}_delta'] = new_cols[c22] - new_cols[c20]

    if {'sanlc_urban_share_2020', 'sanlc_mining_share_2020', 'sanlc_cropland_share_2020'}.issubset(new_cols):
        new_cols['human_land_share_2020'] = (
            new_cols['sanlc_urban_share_2020'].fillna(0.0)
            + new_cols['sanlc_mining_share_2020'].fillna(0.0)
            + new_cols['sanlc_cropland_share_2020'].fillna(0.0)
        )

    if {'sanlc_urban_share_2022', 'sanlc_mining_share_2022', 'sanlc_cropland_share_2022'}.issubset(new_cols):
        new_cols['human_land_share_2022'] = (
            new_cols['sanlc_urban_share_2022'].fillna(0.0)
            + new_cols['sanlc_mining_share_2022'].fillna(0.0)
            + new_cols['sanlc_cropland_share_2022'].fillna(0.0)
        )

    if {'human_land_share_2020', 'human_land_share_2022'}.issubset(new_cols):
        new_cols['human_land_share_delta'] = (
            new_cols['human_land_share_2022'] - new_cols['human_land_share_2020']
        )

    # OSM x SANLC interactions
    if 'osm_pressure_wastewater' in new_cols and 'sanlc_urban_share_2022' in new_cols:
        new_cols['ww_urban'] = (
            new_cols['osm_pressure_wastewater'].fillna(0.0)
            * new_cols['sanlc_urban_share_2022'].fillna(0.0)
        )

    if 'osm_pressure_farm' in new_cols and 'sanlc_cropland_share_2022' in new_cols:
        new_cols['farm_crop'] = (
            new_cols['osm_pressure_farm'].fillna(0.0)
            * new_cols['sanlc_cropland_share_2022'].fillna(0.0)
        )

    if 'osm_pressure_mine' in new_cols and 'sanlc_mining_share_2022' in new_cols:
        new_cols['mine_land'] = (
            new_cols['osm_pressure_mine'].fillna(0.0)
            * new_cols['sanlc_mining_share_2022'].fillna(0.0)
        )

    if new_cols:
        feat_df = pd.DataFrame(new_cols, index=out.index)
        out = pd.concat([out, feat_df], axis=1).copy()  # .copy() defragments memory blocks

    return out


_before_cols = set(df.columns)
df = engineer_features(df)
_new_cols = sorted(set(df.columns) - _before_cols)

print('Feature engineering step completed.')
print('FE_INCLUDE_CLASS_DELTAS:', FE_INCLUDE_CLASS_DELTAS)
print('Added engineered columns:', len(_new_cols))
print('Sample engineered columns:', _new_cols[:40])

## Feature sets

We use two feature sets:

- **FULL_NUMERIC** = numeric non-target features from contract (when present) plus engineered columns
- **C** = compact legacy benchmark (`swir22`, `NDMI`, `MNDWI`, `pet`)


In [ ]:
# Feature sets
# C: legacy benchmark 4-feature set
# FULL_NUMERIC: all numeric non-target features from contract + engineered columns
BENCHMARK_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']

RESERVED_COLS = set(TARGET_COLS + ['spatial_group', 'is_pseudo_valid', 'is_buffer_excluded', 'is_spatial_v2_active'])

# Base from contract (if available)
contract_base = []
if 'contract_cols' in globals() and isinstance(contract_cols, list) and len(contract_cols) > 0:
    contract_base = [c for c in contract_cols if c in df.columns and c not in RESERVED_COLS]

# IMPORTANT: include engineered columns from current dataframe as well.
# This fixes the disconnect where engineered features were not entering PRIMARY_FEATURE_SET.
data_driven_all = [c for c in df.columns if c not in RESERVED_COLS]
base_features = list(dict.fromkeys(contract_base + data_driven_all))

# Current preprocessor is numeric-only, so select numeric subset.
FULL_NUMERIC = [c for c in base_features if pd.api.types.is_numeric_dtype(df[c])]

if not FULL_NUMERIC:
    raise RuntimeError('FULL_NUMERIC feature set is empty. Check input schema/dtypes.')

FEATURE_SETS = {
    'FULL_NUMERIC': FULL_NUMERIC,
    'C': [f for f in BENCHMARK_4 if f in df.columns],
}

PRIMARY_FEATURE_SET = 'FULL_NUMERIC'


def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested


ENGINEERED_EXACT = {
    'is_wet_season',
    'aridity_idx', 'water_balance', 'evap_eff', 'runoff_ratio', 'dry_heat', 'temp_range',
    'rain_wind_event', 'soil_texture_balance', 'soil_fines', 'drainage_proxy',
    'upstream_human_pressure', 'stream_power_proxy', 'river_discharge_per_width',
    'hydro_pressure_index', 'pop_compaction_1km', 'pop_gradient', 'pop_sum_ratio_1_to_5km',
    'human_land_share_2020', 'human_land_share_2022', 'human_land_share_delta',
    'osm_total_pressure', 'farm_rain', 'farm_runoff', 'mine_runoff', 'ww_dry',
    'ww_urban', 'farm_crop', 'mine_land',
}

engineered_cols_in_full = [
    c for c in FULL_NUMERIC
    if c in ENGINEERED_EXACT
    or c.startswith(('month_', 'doy_', 'sanlc_delta_', 'sanlc_abs_delta_'))
    or c.startswith(('osm_log_count_', 'osm_log_dist_', 'osm_inv_dist_', 'osm_local_ratio_', 'osm_ring_count_', 'osm_pressure_', 'osm_near_'))
    or (c.startswith('sanlc_') and ('_share_' in c or '_ratio_' in c or c.endswith('_delta')))
]

print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features')

print('PRIMARY_FEATURE_SET:', PRIMARY_FEATURE_SET)
print('Engineered features included in FULL_NUMERIC:', len(engineered_cols_in_full))
print('Sample engineered-included columns:', engineered_cols_in_full[:40])


## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [ ]:
def get_preprocessor(features_used):
    '''
    Build preprocessing with targeted missing-value policy:
    - SANLC percentage features -> fill missing with 0.0 (absent class)
    - other numeric features -> median imputation
    '''
    sanlc_prefixes = ('sanlc2020_pct_', 'sanlc2022_pct_')
    sanlc_cols = [c for c in features_used if c.startswith(sanlc_prefixes)]
    other_num_cols = [c for c in features_used if c not in sanlc_cols]

    transformers = []

    if other_num_cols:
        other_num_pipe = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num_other', other_num_pipe, other_num_cols))

    if sanlc_cols:
        sanlc_pipe = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num_sanlc', sanlc_pipe, sanlc_cols))

    if not transformers:
        raise RuntimeError('No features provided to preprocessor.')

    return ColumnTransformer(transformers=transformers, remainder='drop')


## Models and shortlist

The active model bank uses tree ensembles only:

- Random Forest (anchor + regularized variants),
- Extra Trees (challenger variants),
- HistGradientBoosting (challenger variants),
- optional log-target wrappers for selected variants.

Optional RFECV can create target-specific feature subsets before the scout/full sweep.


In [ ]:
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor


def log_wrap(model):
    # Apply log transform on targets for optional log-variant.
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )


# Anchor RF defaults
DEFAULT_RF_PARAMS = {
    'n_estimators': 600,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

# Challenger ET defaults
DEFAULT_ET_PARAMS = {
    'n_estimators': 700,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

# Challenger HGB defaults (no n_jobs in this estimator)
DEFAULT_HGB_PARAMS = {
    'max_depth': 8,
    'learning_rate': 0.05,
    'max_iter': 450,
    'min_samples_leaf': 25,
    'l2_regularization': 0.0,
    'random_state': 42,
}

MODEL_SPECS = {
    # RF anchor + regularized ladder
    'RF_n600_raw': {'kind': 'rf', 'params': {}, 'log_target': False},
    'RF_n600_Log': {'kind': 'rf', 'params': {}, 'log_target': True},
    'RF_regA_raw': {
        'kind': 'rf',
        'params': {'max_features': 0.35, 'min_samples_leaf': 10, 'min_samples_split': 20, 'max_depth': 16},
        'log_target': False,
    },
    'RF_regB_raw': {
        'kind': 'rf',
        'params': {'max_features': 0.20, 'min_samples_leaf': 20, 'min_samples_split': 50, 'max_depth': 12},
        'log_target': False,
    },
    'RF_regA_Log': {
        'kind': 'rf',
        'params': {'max_features': 0.35, 'min_samples_leaf': 10, 'min_samples_split': 20, 'max_depth': 16},
        'log_target': True,
    },

    # ET challenger + regularized
    'ET_n700_raw': {'kind': 'et', 'params': {}, 'log_target': False},
    'ET_n700_Log': {'kind': 'et', 'params': {}, 'log_target': True},
    'ET_regA_raw': {
        'kind': 'et',
        'params': {'max_features': 0.25, 'min_samples_leaf': 20, 'min_samples_split': 50, 'max_depth': 14},
        'log_target': False,
    },

    # HGB challenger + regularized ladder
    'HGB_n450_raw': {'kind': 'hgb', 'params': {}, 'log_target': False},
    'HGB_n450_Log': {'kind': 'hgb', 'params': {}, 'log_target': True},
    'HGB_regA_raw': {
        'kind': 'hgb',
        'params': {
            'learning_rate': 0.03,
            'max_leaf_nodes': 15,
            'min_samples_leaf': 80,
            'l2_regularization': 5.0,
            'early_stopping': True,
            'validation_fraction': 0.15,
            'n_iter_no_change': 30,
            'max_iter': 450,
        },
        'log_target': False,
    },
    'HGB_regB_raw': {
        'kind': 'hgb',
        'params': {
            'learning_rate': 0.05,
            'max_leaf_nodes': 31,
            'min_samples_leaf': 40,
            'l2_regularization': 1.0,
            'early_stopping': True,
            'validation_fraction': 0.15,
            'n_iter_no_change': 25,
            'max_iter': 450,
        },
        'log_target': False,
    },
}


def build_model_from_spec(spec):
    kind = spec['kind']
    params = spec.get('params', {})
    use_log = spec.get('log_target', True)

    if kind == 'rf':
        p = DEFAULT_RF_PARAMS.copy()
        p.update(params)
        base = RandomForestRegressor(**p)
    elif kind == 'et':
        p = DEFAULT_ET_PARAMS.copy()
        p.update(params)
        base = ExtraTreesRegressor(**p)
    elif kind == 'hgb':
        p = DEFAULT_HGB_PARAMS.copy()
        p.update(params)
        base = HistGradientBoostingRegressor(**p)
    else:
        raise ValueError(f'Unknown model kind: {kind}')

    return log_wrap(base) if use_log else base


MODEL_BANK = {
    name: build_model_from_spec(spec)
    for name, spec in tqdm(
        MODEL_SPECS.items(),
        total=len(MODEL_SPECS),
        desc='Building model bank'
    )
}

# Compact target-wise sweep to keep runtime under control while adding regularization.
TARGET_MODEL_ORDER = {
    'Total Alkalinity': [
        'RF_n600_raw', 'RF_regA_raw', 'RF_regB_raw',
        'ET_n700_raw', 'ET_regA_raw',
        'HGB_n450_raw', 'HGB_regA_raw', 'HGB_regB_raw',
    ],
    'Electrical Conductance': [
        'RF_n600_raw', 'RF_regA_raw', 'RF_regB_raw',
        'ET_n700_raw', 'ET_regA_raw',
        'HGB_n450_raw', 'HGB_regA_raw', 'HGB_regB_raw',
    ],
    'Dissolved Reactive Phosphorus': [
        'RF_n600_raw', 'RF_n600_Log', 'RF_regA_raw', 'RF_regA_Log',
        'ET_n700_raw', 'ET_n700_Log',
        'HGB_n450_raw', 'HGB_n450_Log', 'HGB_regA_raw',
    ],
}

# -------------------------
# Optional RFECV feature selection (spatial-group aware)
# -------------------------
RFECV_ENABLED = True
RFECV_STEP = 20
RFECV_MIN_FEATURES = 60
RFECV_N_JOBS = -1
RFECV_SCORING = 'r2'

RFECV_TARGET_FEATURE_SET = {t: PRIMARY_FEATURE_SET for t in TARGET_COLS}

if RFECV_ENABLED:
    from sklearn.feature_selection import RFECV

    base_fs = FEATURE_SETS[PRIMARY_FEATURE_SET]
    train_mask = (~df['is_pseudo_valid']) & (df['is_spatial_v2_active'])
    X_train_full = df.loc[train_mask, base_fs].copy()
    groups_train = df.loc[train_mask, 'spatial_group'].astype(str).copy()

    n_groups = int(groups_train.nunique())
    n_splits = min(5, max(2, n_groups))

    rfecv_estimator = RandomForestRegressor(
        n_estimators=500,
        max_features=0.35,
        min_samples_leaf=10,
        min_samples_split=20,
        max_depth=16,
        random_state=42,
        n_jobs=-1,
    )

    print(f'RFECV enabled | n_base_features={len(base_fs)} | n_groups_train={n_groups} | n_splits={n_splits}')

    outer_bar = tqdm(TARGET_COLS, total=len(TARGET_COLS), desc='RFECV targets', position=0)

    for target in outer_bar:
        outer_bar.set_postfix_str(target)

        y_train = df.loc[train_mask, target].astype(float).copy()

        with tqdm(
            total=1,
            desc=f'Fitting RFECV: {target}',
            leave=False,
            position=1
        ) as inner_bar:
            selector = RFECV(
                estimator=clone(rfecv_estimator),
                step=RFECV_STEP,
                min_features_to_select=min(RFECV_MIN_FEATURES, len(base_fs)),
                cv=GroupKFold(n_splits=n_splits),
                scoring=RFECV_SCORING,
                n_jobs=RFECV_N_JOBS,
            )
            selector.fit(X_train_full, y_train, groups=groups_train)
            inner_bar.update(1)

        selected_cols = [f for f, keep in zip(base_fs, selector.support_) if keep]
        if len(selected_cols) == 0:
            selected_cols = list(base_fs)

        fs_name = f'RFECV_{target_key(target)}'
        FEATURE_SETS[fs_name] = selected_cols
        RFECV_TARGET_FEATURE_SET[target] = fs_name

        best_cv = float(np.nanmax(selector.cv_results_["mean_test_score"]))
        print(
            f'RFECV {target}: selected={len(selected_cols)} / {len(base_fs)} '
            f'| best_cv={best_cv:.6f} | fs_name={fs_name}'
        )
else:
    print('RFECV disabled (RFECV_ENABLED=False). Using PRIMARY_FEATURE_SET for all targets.')

TARGET_SWEEP = {
    target: [
        (RFECV_TARGET_FEATURE_SET.get(target, PRIMARY_FEATURE_SET), model_name)
        for model_name in tqdm(
            TARGET_MODEL_ORDER[target],
            total=len(TARGET_MODEL_ORDER[target]),
            desc=f'Assembling sweep for {target}',
            leave=False
        )
    ]
    for target in tqdm(TARGET_COLS, total=len(TARGET_COLS), desc='Building target sweep')
}

print('Model sweep active: regularization ladder + RF baseline anchor')

sweep_rows = []
for t in tqdm(TARGET_COLS, total=len(TARGET_COLS), desc='Preparing sweep table', leave=False):
    recipes = TARGET_SWEEP[t]
    for fs, m in tqdm(recipes, total=len(recipes), desc=f'{t} rows', leave=False):
        sweep_rows.append((t, fs, m))

display(pd.DataFrame(
    sweep_rows,
    columns=['target', 'feature_set', 'model']
))

## Grouped evaluation helpers

This section runs grouped OOF evaluation with `GroupKFold` on `spatial_group`, plus:

- a matched dummy-median baseline,
- pseudo-holdout scoring,
- robustness metrics (mean/min fold R2),
- artifact saving for finalist pipelines.


In [ ]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None, show_fold_progress=False, fold_desc=None):
    # Run grouped OOF evaluation with GroupKFold on spatial clusters,
    # plus a dummy median baseline on exactly the same folds/holdout.
    d = df_local.copy()

    if 'spatial_group' not in d.columns:
        raise RuntimeError(
            'Missing `spatial_group` in df_local. Run the data-loading/split setup cell (cell 8) '
            'to create spatial groups and pseudo-holdout flags before scout/full stages.'
        )

    if allowed_regions is not None:
        allowed = set(pd.Series(allowed_regions).astype(str).tolist())
        d = d[d['spatial_group'].astype(str).isin(allowed)].copy()

    if 'is_spatial_v2_active' in d.columns:
        d = d[d['is_spatial_v2_active'].astype(bool)].copy()
        if d.empty:
            raise RuntimeError('No active rows remain after applying spatial v2 buffer exclusion.')

    X = d[features_used].reset_index(drop=True)
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['spatial_group'].astype(str).reset_index(drop=True)

    n_groups = int(groups.nunique())
    if n_groups < 2:
        raise RuntimeError('Need at least 2 spatial groups for grouped CV.')

    n_splits = min(int(CV_N_SPLITS), n_groups)
    gkf = GroupKFold(n_splits=n_splits)

    pred = np.full(len(d), np.nan, dtype=float)
    dummy_pred = np.full(len(d), np.nan, dtype=float)

    fold_rows = []
    dummy_fold_rows = []

    fold_iterator = tqdm(
        gkf.split(X, y, groups=groups),
        total=n_splits,
        desc=(fold_desc or f'CV {target}'),
        leave=False,
        disable=not show_fold_progress,
        unit='fold'
    )

    for fold_id, (train_idx, test_idx) in enumerate(fold_iterator, start=1):
        pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])

        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        fold_pred = np.asarray(pipe.predict(X_te), dtype=float)
        pred[test_idx] = fold_pred

        fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, fold_pred))),
            'mae': float(mean_absolute_error(y_te, fold_pred)),
        })

        dummy_value = float(np.median(y_tr))
        dummy_fold_pred = np.full(len(test_idx), dummy_value, dtype=float)
        dummy_pred[test_idx] = dummy_fold_pred

        dummy_fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, dummy_fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, dummy_fold_pred))),
            'mae': float(mean_absolute_error(y_te, dummy_fold_pred)),
        })

    if np.isnan(pred).any():
        raise RuntimeError('OOF predictions contain NaN values.')
    if np.isnan(dummy_pred).any():
        raise RuntimeError('Dummy OOF predictions contain NaN values.')

    fold_df = pd.DataFrame(fold_rows)
    dummy_fold_df = pd.DataFrame(dummy_fold_rows)

    holdout_r2 = np.nan
    dummy_holdout_r2 = np.nan

    if 'is_pseudo_valid' in d.columns:
        hold_mask = d['is_pseudo_valid'].astype(bool).reset_index(drop=True)

        if hold_mask.any() and int((~hold_mask).sum()) >= 2 and int(hold_mask.sum()) >= 2:
            hold_train_groups = set(groups.loc[~hold_mask].tolist())
            hold_test_groups = set(groups.loc[hold_mask].tolist())

            if hold_train_groups.intersection(hold_test_groups):
                raise RuntimeError('Pseudo-holdout leakage detected: train and holdout share groups.')

            hold_pipe = Pipeline([
                ('preprocessor', get_preprocessor(features_used)),
                ('model', clone(estimator))
            ])

            hold_pipe.fit(X.loc[~hold_mask], y.loc[~hold_mask])
            hold_pred = np.asarray(hold_pipe.predict(X.loc[hold_mask]), dtype=float)
            holdout_r2 = float(r2_score(y.loc[hold_mask], hold_pred))

            dummy_hold_value = float(np.median(y.loc[~hold_mask]))
            dummy_hold_pred = np.full(int(hold_mask.sum()), dummy_hold_value, dtype=float)
            dummy_holdout_r2 = float(r2_score(y.loc[hold_mask], dummy_hold_pred))

    model_r2 = float(r2_score(y, pred))
    model_rmse = float(np.sqrt(mean_squared_error(y, pred)))
    model_mae = float(mean_absolute_error(y, pred))
    model_mean_fold_r2 = float(fold_df['r2'].mean()) if not fold_df.empty else np.nan
    model_min_fold_r2 = float(fold_df['r2'].min()) if not fold_df.empty else np.nan

    dummy_r2 = float(r2_score(y, dummy_pred))
    dummy_rmse = float(np.sqrt(mean_squared_error(y, dummy_pred)))
    dummy_mae = float(mean_absolute_error(y, dummy_pred))
    dummy_mean_fold_r2 = float(dummy_fold_df['r2'].mean()) if not dummy_fold_df.empty else np.nan
    dummy_min_fold_r2 = float(dummy_fold_df['r2'].min()) if not dummy_fold_df.empty else np.nan

    delta_holdout = np.nan
    if not pd.isna(holdout_r2) and not pd.isna(dummy_holdout_r2):
        delta_holdout = float(holdout_r2 - dummy_holdout_r2)

    return {
        'pred': pred,
        'rmse': model_rmse,
        'mae': model_mae,
        'r2': model_r2,
        'mean_fold_r2': model_mean_fold_r2,
        'min_fold_r2': model_min_fold_r2,
        'holdout_r2': holdout_r2,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': n_groups,
        'dummy_r2': dummy_r2,
        'dummy_rmse': dummy_rmse,
        'dummy_mae': dummy_mae,
        'dummy_mean_fold_r2': dummy_mean_fold_r2,
        'dummy_min_fold_r2': dummy_min_fold_r2,
        'dummy_holdout_r2': dummy_holdout_r2,
        'delta_r2_vs_dummy': float(model_r2 - dummy_r2),
        'delta_min_fold_r2_vs_dummy': float(model_min_fold_r2 - dummy_min_fold_r2),
        'delta_holdout_r2_vs_dummy': delta_holdout,
    }


GLOBAL_R2_WEIGHT = 0.60
HOLDOUT_R2_WEIGHT = 0.25
MIN_FOLD_R2_WEIGHT = 0.15


def compute_selection_score(overall_r2, holdout_r2, min_fold_r2=None):
    # Build a finalist selection score with holdout awareness and fold robustness.
    holdout_term = 0.0 if pd.isna(holdout_r2) else float(holdout_r2)
    min_fold_term = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)

    return float(
        GLOBAL_R2_WEIGHT * float(overall_r2) +
        HOLDOUT_R2_WEIGHT * holdout_term +
        MIN_FOLD_R2_WEIGHT * min_fold_term
    )


def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    # Fit the final full-data pipeline and save preprocessor + model artifacts.
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path



## Stage 1: Scout run

Run a broad target-wise scout sweep over `(feature_set, model)` recipes and log metrics to MLflow.
By default, scout runs on all spatial groups.


In [ ]:
SCOUT_GROUPS = None
print('Scout scope: ALL spatial groups')

rows_scout = []

for target in tqdm(TARGET_COLS, desc='Scout targets', unit='target'):
    recipes = TARGET_SWEEP[target]
    for feature_set_name, model_name in tqdm(recipes, desc=f'Scout {target}', unit='model', leave=False):
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print()
        print(f'--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_GROUPS,
                show_fold_progress=True,
                fold_desc=f'Scout CV {target_key(target)}'
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('holdout_r2', out['holdout_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('dummy_r2', out['dummy_r2'])
            mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
            mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
            if not pd.isna(out['delta_holdout_r2_vs_dummy']):
                mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
            mlflow.log_metric('cv_time_sec', dt)

            selection_score = compute_selection_score(
                overall_r2=out['r2'],
                holdout_r2=out['holdout_r2'],
                min_fold_r2=out['min_fold_r2']
            )

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'holdout_r2': out['holdout_r2'],
                'dummy_r2': out['dummy_r2'],
                'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
                'dummy_min_fold_r2': out['dummy_min_fold_r2'],
                'dummy_holdout_r2': out['dummy_holdout_r2'],
                'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
                'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
                'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
                'selection_score': selection_score,
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
            f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
            f"selection_score={selection_score:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Scout results:')
display(scout_df)



## Stage 2: Full finalists

Build a finalist union shortlist per target (selection score, global R2, and robustness views),
then run full grouped CV and save artifacts for each finalist run.


In [ ]:
def build_finalist_shortlist(scout_df_local, top_n=2):
    # Keep a union of top candidates by complementary views so we do not
    # discard globally stronger or more robust models too early.
    pieces = []

    for target_name, tdf in scout_df_local.groupby('target'):
        by_selection = tdf.sort_values(
            ['selection_score', 'r2', 'min_fold_r2'],
            ascending=[False, False, False]
        ).head(top_n)

        by_global = tdf.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(top_n)

        by_robust = tdf.sort_values(
            ['min_fold_r2', 'r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(1)

        shortlist = pd.concat([by_selection, by_global, by_robust], axis=0)
        shortlist = shortlist.drop_duplicates(subset=['run_name']).reset_index(drop=True)
        pieces.append(shortlist)

    return pd.concat(pieces, axis=0).reset_index(drop=True)


finalist_df = build_finalist_shortlist(scout_df, top_n=2)

print('Finalists (union shortlist):')
display(finalist_df[[
    'target',
    'feature_set',
    'model_name',
    'selection_score',
    'holdout_r2',
    'r2',
    'min_fold_r2',
    'delta_r2_vs_dummy',
    'delta_holdout_r2_vs_dummy',
    'run_name'
]])

rows_full = []
OOF_PREDS = {}

for _, row in tqdm(finalist_df.iterrows(), total=len(finalist_df), desc='Full finalists', unit='run'):
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print()
    print(f'=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None,
            show_fold_progress=True,
            fold_desc=f'Full CV {target_key(target)}'
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('holdout_r2', out['holdout_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('dummy_r2', out['dummy_r2'])
        mlflow.log_metric('dummy_holdout_r2', out['dummy_holdout_r2'])
        mlflow.log_metric('delta_r2_vs_dummy', out['delta_r2_vs_dummy'])
        if not pd.isna(out['delta_holdout_r2_vs_dummy']):
            mlflow.log_metric('delta_holdout_r2_vs_dummy', out['delta_holdout_r2_vs_dummy'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            holdout_r2=out['holdout_r2'],
            min_fold_r2=out['min_fold_r2']
        )

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'holdout_r2': out['holdout_r2'],
            'dummy_r2': out['dummy_r2'],
            'dummy_mean_fold_r2': out['dummy_mean_fold_r2'],
            'dummy_min_fold_r2': out['dummy_min_fold_r2'],
            'dummy_holdout_r2': out['dummy_holdout_r2'],
            'delta_r2_vs_dummy': out['delta_r2_vs_dummy'],
            'delta_min_fold_r2_vs_dummy': out['delta_min_fold_r2_vs_dummy'],
            'delta_holdout_r2_vs_dummy': out['delta_holdout_r2_vs_dummy'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} vs dummy={out['dummy_r2']:.4f} (delta={out['delta_r2_vs_dummy']:.4f}) | "
        f"Holdout R2={out['holdout_r2']:.4f} vs dummy={out['dummy_holdout_r2']:.4f} | "
        f"selection_score={selection_score:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f} vs dummy={out['dummy_min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print()
print('Full results:')
display(full_df)



## Freeze Manifest A and Manifest B

This stage freezes production manifests from full-stage results:

- **Manifest A**: safety-first anchor per target,
- **Manifest B**: modest challenger path where beneficial.

Selection prioritizes pseudo-holdout improvement versus dummy baseline.


In [ ]:
# OPTIONAL LEGACY CONSTANTS (currently unused by active manifest logic)
# TARGET_GLOBAL_R2_FLOOR = {
#     'Total Alkalinity': 0.00,
#     'Electrical Conductance': 0.00,
#     'Dissolved Reactive Phosphorus': -0.20,
# }
#
# DRP_SAFE_MODELS = [
#     'RF_n600_raw',
#     'RF_n600_Log',
# ]


In [ ]:
manifest_A = {}
manifest_B = {}


def pick_with_gate(tdf, preferred_models=None):
    # Hard gate: prefer rows that beat dummy on pseudo-holdout.
    gated = tdf[tdf['delta_holdout_r2_vs_dummy'] > 0].copy()
    pool = gated if not gated.empty else tdf.copy()

    if preferred_models:
        for model_name in preferred_models:
            pref = pool[pool['model_name'] == model_name].copy()
            if not pref.empty:
                pool = pref
                break

    pool = pool.sort_values(
        ['delta_holdout_r2_vs_dummy', 'holdout_r2', 'selection_score', 'r2'],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    if pool.empty:
        raise RuntimeError('No candidate rows available after gating/sorting.')
    return pool.iloc[0]


for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy().reset_index(drop=True)

    if tdf.empty:
        raise RuntimeError(f'No full-stage results available for target: {target}. Run Stage 2 first and verify full_df.')

    # Manifest A = safety anchor (RF-first preference)
    if target == 'Total Alkalinity':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_raw', 'RF_n600_Log'])
        chosen_B = pick_with_gate(tdf, preferred_models=None)

    elif target == 'Electrical Conductance':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_raw', 'RF_n600_Log'])
        chosen_B = pick_with_gate(tdf, preferred_models=None)

    elif target == 'Dissolved Reactive Phosphorus':
        chosen_A = pick_with_gate(tdf, preferred_models=['RF_n600_Log', 'RF_n600_raw'])
        chosen_B = chosen_A

    manifest_A[target] = chosen_A.to_dict()
    manifest_B[target] = chosen_B.to_dict()


DRP_DELTA_HOLDOUT = float(manifest_A['Dissolved Reactive Phosphorus']['delta_holdout_r2_vs_dummy'])
DRP_USE_MODEL = DRP_DELTA_HOLDOUT > 0.0

print('DRP_USE_MODEL:', DRP_USE_MODEL, '| delta_holdout_vs_dummy:', round(DRP_DELTA_HOLDOUT, 6))

print('=== MANIFEST A (GATED, RF-ANCHORED) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_A.items()
}, indent=2))

print('\n=== MANIFEST B (GATED, BEST-AVAILABLE CHALLENGER) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'dummy_holdout_r2': float(v['dummy_holdout_r2']) if pd.notna(v['dummy_holdout_r2']) else None,
        'delta_holdout_r2_vs_dummy': float(v['delta_holdout_r2_vs_dummy']) if pd.notna(v['delta_holdout_r2_vs_dummy']) else None,
        'selection_score': float(v['selection_score']),
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_B.items()
}, indent=2))


## Diagnostic For Retraining Before Making Submission

In [ ]:
# OPTIONAL DIAGNOSTIC (disabled for faster runs)
# diag_feature_set = FEATURE_SETS.get(PRIMARY_FEATURE_SET, FEATURE_SETS.get('C', ['swir22', 'NDMI', 'MNDWI', 'pet']))
# cols = [c for c in diag_feature_set if c in df.columns] + TARGET_COLS
# display(df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).T)


## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [ ]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using saved preprocessor/model artifacts.
    Raises clear errors when artifacts or features are missing.
    '''
    feats = json.loads(entry['features_used_json'])
    preproc_path = entry['preproc_path']
    model_path = entry['model_path']

    if not os.path.exists(preproc_path):
        raise FileNotFoundError(f'Missing preprocessor artifact: {preproc_path}')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Missing model artifact: {model_path}')

    missing_feats = [f for f in feats if f not in df_val_local.columns]
    if missing_feats:
        raise RuntimeError(f'Missing validation features for manifest run {entry.get("run_name", "unknown")}: {missing_feats[:20]}')

    pre = joblib.load(preproc_path)
    mdl = joblib.load(model_path)

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)


## Build Shot A / B / C / D

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B
- **Shot D** = robust DRP soft-hedge (always blends a little model signal)


In [ ]:
df_val = pd.read_parquet(VALID_PATH).copy()
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

if len(df_val) != len(tpl):
    raise RuntimeError(f'Validation rows ({len(df_val)}) do not match submission template rows ({len(tpl)}).')

# Validate that all needed features exist in validation
needed_feats = set()
for t in ['Total Alkalinity', 'Electrical Conductance']:
    needed_feats.update(json.loads(manifest_A[t]['features_used_json']))
    needed_feats.update(json.loads(manifest_B[t]['features_used_json']))

# Shot D always uses a DRP model hedge, even when DRP_USE_MODEL is False.
USE_DRP_MODEL_IN_SHOTD = True
if DRP_USE_MODEL or USE_DRP_MODEL_IN_SHOTD:
    needed_feats.update(json.loads(manifest_A['Dissolved Reactive Phosphorus']['features_used_json']))

missing_feats = sorted([f for f in needed_feats if f not in df_val.columns])
if missing_feats:
    raise RuntimeError(f'Missing validation features: {missing_feats}')


def clip_by_train_quantile(pred, target, q_hi=0.995):
    hi = float(df[target].quantile(q_hi))
    return np.clip(np.asarray(pred, dtype=float), 0, hi)


drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())

# -------------------------
# Shot A: safe anchor
# -------------------------
shotA = tpl.copy()

pred_ta_a = predict_from_manifest_entry(manifest_A['Total Alkalinity'], df_val)
pred_ec_a = predict_from_manifest_entry(manifest_A['Electrical Conductance'], df_val)

shotA['Total Alkalinity'] = clip_by_train_quantile(pred_ta_a, 'Total Alkalinity')
shotA['Electrical Conductance'] = clip_by_train_quantile(pred_ec_a, 'Electrical Conductance')

if DRP_USE_MODEL:
    drp_a = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    shotA['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_a, 'Dissolved Reactive Phosphorus')
    print('Shot A DRP model:', manifest_A['Dissolved Reactive Phosphorus']['run_name'])
else:
    shotA['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot A DRP fallback: train median')

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# -------------------------
# Shot B: modest challenger
# -------------------------
shotB = tpl.copy()
shotB['Total Alkalinity'] = shotA['Total Alkalinity']

if manifest_B['Electrical Conductance']['run_name'] != manifest_A['Electrical Conductance']['run_name']:
    ec_safe = shotA['Electrical Conductance'].values
    ec_chal = predict_from_manifest_entry(manifest_B['Electrical Conductance'], df_val)
    ec_blend = 0.35 * ec_safe + 0.65 * ec_chal
    shotB['Electrical Conductance'] = clip_by_train_quantile(ec_blend, 'Electrical Conductance')
    print('Shot B EC challenger blend:', manifest_B['Electrical Conductance']['run_name'])
else:
    shotB['Electrical Conductance'] = shotA['Electrical Conductance']
    print('Shot B EC kept from Manifest A')

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend = 0.20 * drp_model + 0.80 * drp_train_median
    shotB['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend, 'Dissolved Reactive Phosphorus')
    print('Shot B DRP model+median alpha=0.20')
else:
    shotB['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot B DRP fallback: train median')

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# -------------------------
# Shot C: stronger DRP hedge
# -------------------------
shotC = tpl.copy()
shotC['Total Alkalinity'] = clip_by_train_quantile(
    0.85 * shotA['Total Alkalinity'] + 0.15 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotC['Electrical Conductance'] = clip_by_train_quantile(
    0.50 * shotA['Electrical Conductance'] + 0.50 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)

if DRP_USE_MODEL:
    drp_model = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
    drp_blend_c = 0.10 * drp_model + 0.90 * drp_train_median
    shotC['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend_c, 'Dissolved Reactive Phosphorus')
    print('Shot C DRP model+median alpha=0.10')
else:
    shotC['Dissolved Reactive Phosphorus'] = drp_train_median
    print('Shot C DRP fallback: train median')

assert_submission_integrity(shotC, tpl, TARGET_COLS)

# -------------------------
# Shot D: robust DRP soft-hedge
# -------------------------
shotD = tpl.copy()
shotD['Total Alkalinity'] = clip_by_train_quantile(
    0.90 * shotA['Total Alkalinity'] + 0.10 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotD['Electrical Conductance'] = clip_by_train_quantile(
    0.70 * shotA['Electrical Conductance'] + 0.30 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)

drp_model_d = predict_from_manifest_entry(manifest_A['Dissolved Reactive Phosphorus'], df_val)
if DRP_USE_MODEL:
    drp_alpha_d = 0.25
else:
    drp_alpha_d = 0.05

drp_blend_d = drp_alpha_d * drp_model_d + (1.0 - drp_alpha_d) * drp_train_median
shotD['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(drp_blend_d, 'Dissolved Reactive Phosphorus')
print(f'Shot D DRP model+median alpha={drp_alpha_d:.2f} (soft hedge always on)')

assert_submission_integrity(shotD, tpl, TARGET_COLS)

# -------------------------
# Save
# -------------------------
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_challenger_blend.csv'
pathC = f'../data/submission/submission_{stamp}_C_hedge.csv'
pathD = f'../data/submission/submission_{stamp}_D_drp_softhedge.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)
shotD.drop(columns=['row_id']).to_csv(pathD, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)
print('D:', pathD)


## Submission diagnostics

Optional diagnostics for submission variants are provided in the final code cell and are disabled by default for faster runs.


In [ ]:
# OPTIONAL SUBMISSION DIAGNOSTICS (disabled for faster runs)
# def summarize_shot(shot_df, name):
#     print(f'\n{name} stats')
#     stats = shot_df[TARGET_COLS].describe(
#         percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
#     ).T
#     display(stats[['min', '1%', '5%', '50%', 'mean', '95%', '99%', 'max']])
#
# summarize_shot(shotA, 'Shot A')
# summarize_shot(shotB, 'Shot B')
# summarize_shot(shotC, 'Shot C')
# summarize_shot(shotD, 'Shot D')
#
# print('\nMean absolute deltas vs Shot A')
# delta_tbl = pd.DataFrame({
#     'target': TARGET_COLS,
#     'B_vs_A_mae': [float(np.mean(np.abs(shotB[t] - shotA[t]))) for t in TARGET_COLS],
#     'C_vs_A_mae': [float(np.mean(np.abs(shotC[t] - shotA[t]))) for t in TARGET_COLS],
#     'D_vs_A_mae': [float(np.mean(np.abs(shotD[t] - shotA[t]))) for t in TARGET_COLS],
# })
# display(delta_tbl)
#
# tracker = pd.DataFrame([
#     {'file': pathA, 'hypothesis': 'Safety anchor (lowest variance)'},
#     {'file': pathD, 'hypothesis': 'DRP soft hedge (tiny model signal + median)'},
#     {'file': pathC, 'hypothesis': 'Balanced hedge between A and B'},
#     {'file': pathB, 'hypothesis': 'Most aggressive on EC challenger blend'},
# ])
#
# print('\nSubmission tracker:')
# display(tracker)
#
# print('\nSuggested upload order: A -> D -> C -> B')
